# Stage-Wise Curvature Tuning on DTD

## Environment Setup

This notebook is designed to run independently in Google Colab.

The repository is cloned or updated, the `stage-wise-ct` branch is selected, and the package versions used for the research experiments are installed.

After the dependency installation, the Colab runtime must be restarted before continuing with the experiment cells.

In [ ]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

In [ ]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

### Runtime Restart Required

After installing the dependencies and NumPy 1.26.3, restart the Colab runtime so that the required package versions are loaded correctly.

**After restarting, do not rerun the installation cells above. Continue from the next cell.**

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

In [ ]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

# Cross-Dataset Evaluation: DTD

The previous experiments were performed on Beans. To determine whether the behavior of Stage-Wise Curvature Tuning generalizes beyond a single dataset, the same method is now evaluated on the Describable Textures Dataset (DTD).

Importantly, the main SW-CT hyperparameters selected during the Beans experiments are kept fixed:

- Initial β = **0.78**
- β learning rate = **0.01**
- \(c = 0.5\)
- Four trainable stage-wise β parameters

The goal is not to retune SW-CT specifically for DTD, but to test whether the method and the learned stage-wise curvature structure transfer to a different visual classification task.

In [ ]:
%cd /content/curvature-tuning-research/src/curvature-tuning

from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_dtd",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

/content/curvature-tuning-research/src/curvature-tuning


100%|██████████| 625M/625M [00:54<00:00, 11.5MB/s]


Train samples: 1880
Validation samples: 1880
Test samples: 1880
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Label min: 1
Label max: 45


## Original Baseline and S-CT on DTD

The original Baseline and Single-Parameter Curvature Tuning method are evaluated on DTD using seeds 42, 43, and 44.

S-CT performs its original validation-based search over a single global β value. These experiments provide the reference against which Stage-Wise CT will be compared.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-23 23:40:41.907 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed42.log
2026-09-23 23:40:41.910 | INFO     | __main__:main:58 - Running on cuda
2026-09-23 23:40:42.452 | INFO     | __main__:main:85 - Testing baseline...
2026-09-23 23:40:42.466 | INFO     | __main__:main:88 - Number of trainable parameters: 24111
2026-09-23 23:40:42.466 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-23 23:41:04.923 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.976914, Accuracy: 0.00%
2026-09-23 23:41:05.262 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.015505, Val Accuracy: 32.07%
2026-09-23 23:41:05.263 | INFO     | train:linear_probe:190 - New best validation accuracy: 32.07 at epoch 1
2026-09-23 23:41:05.311 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.898652, Accuracy: 43.75%
2026-09-23 23:41:05.622 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.050765, Val Accuracy: 50.96%

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-23 23:57:20.373 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed43.log
2026-09-23 23:57:20.375 | INFO     | __main__:main:58 - Running on cuda
2026-09-23 23:57:21.111 | INFO     | __main__:main:85 - Testing baseline...
2026-09-23 23:57:21.133 | INFO     | __main__:main:88 - Number of trainable parameters: 24111
2026-09-23 23:57:21.134 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-23 23:57:42.015 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.055932, Accuracy: 0.00%
2026-09-23 23:57:42.374 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.985955, Val Accuracy: 30.59%
2026-09-23 23:57:42.375 | INFO     | train:linear_probe:190 - New best validation accuracy: 30.59 at epoch 1
2026-09-23 23:57:42.422 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.764283, Accuracy: 31.25%
2026-09-23 23:57:42.734 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.042676, Val Accuracy: 51.49%

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-24 00:13:52.884 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed44.log
2026-09-24 00:13:52.887 | INFO     | __main__:main:58 - Running on cuda
2026-09-24 00:13:53.414 | INFO     | __main__:main:85 - Testing baseline...
2026-09-24 00:13:53.427 | INFO     | __main__:main:88 - Number of trainable parameters: 24111
2026-09-24 00:13:53.427 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-24 00:14:13.961 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.173090, Accuracy: 0.00%
2026-09-24 00:14:14.291 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.030038, Val Accuracy: 31.22%
2026-09-24 00:14:14.292 | INFO     | train:linear_probe:190 - New best validation accuracy: 31.22 at epoch 1
2026-09-24 00:14:14.352 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.947498, Accuracy: 37.50%
2026-09-24 00:14:14.688 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.067236, Val Accuracy: 51.12%

## 4-Parameter SW-CT on DTD

The fixed 4-parameter SW-CT configuration developed on Beans is now evaluated on DTD using the same three seeds.

No DTD-specific β initialization or β learning-rate tuning is performed. In addition to test accuracy, the learned β values from each network stage are recorded to determine whether the stage-wise curvature pattern observed on Beans also appears on DTD.

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --transfer_train_bs 32 \
    --transfer_test_bs 64

2026-09-24 00:30:45.069 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed42_initbeta0.78_betalr0.01.log
2026-09-24 00:30:45.128 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-24 00:30:45.626 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-24 00:30:45.627 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-24 00:30:45.627 | INFO     | __main__:main:333 - Total trainable parameters: 24115
2026-09-24 00:30:46.861 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.106285, Accuracy: 9.38%
2026-09-24 00:31:10.376 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.855677, Val Accuracy: 29.10%
2026-09-24 00:31:10.377 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=29.10, betas=[0.779, 0.7996, 0.779, 0.7556]
2026-09-24 00:31:10.392 | INFO     | __main_

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --transfer_train_bs 32 \
    --transfer_test_bs 64

2026-09-24 00:39:06.488 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed43_initbeta0.78_betalr0.01.log
2026-09-24 00:39:06.548 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-24 00:39:07.042 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-24 00:39:07.043 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-24 00:39:07.043 | INFO     | __main__:main:333 - Total trainable parameters: 24115
2026-09-24 00:39:08.313 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.396770, Accuracy: 0.00%
2026-09-24 00:39:31.705 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.834445, Val Accuracy: 28.24%
2026-09-24 00:39:31.706 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=28.24, betas=[0.7938, 0.7946, 0.7891, 0.7441]
2026-09-24 00:39:31.733 | INFO     | __mai

In [ ]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --transfer_train_bs 32 \
    --transfer_test_bs 64

2026-09-24 00:47:25.111 | INFO     | __main__:main:245 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed44_initbeta0.78_betalr0.01.log
2026-09-24 00:47:25.221 | INFO     | __main__:main:261 - Running on cuda
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-09-24 00:47:25.863 | INFO     | __main__:main:312 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-09-24 00:47:25.863 | INFO     | __main__:main:328 - Trainable curvature parameters: 4
2026-09-24 00:47:25.864 | INFO     | __main__:main:333 - Total trainable parameters: 24115
2026-09-24 00:47:27.038 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.886103, Accuracy: 9.38%
2026-09-24 00:47:49.460 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.845332, Val Accuracy: 29.47%
2026-09-24 00:47:49.461 | INFO     | __main__:transfer:127 - Epoch 1: val_acc=29.47, betas=[0.8043, 0.7741, 0.7865, 0.7407]
2026-09-24 00:47:49.480 | INFO     | __mai

In [ ]:
import json
import glob
import numpy as np
import os

seeds = [42, 43, 44]

accs = []
val_accs = []
betas = []

for seed in seeds:
    # Finds the result even if the filename format changed slightly
    pattern = (
        f"results/stage_ct_imagenet_to_dtd_"
        f"resnet18_seed{seed}*beta0.78*betalr0.01*.json"
    )

    matches = glob.glob(pattern)

    if not matches:
        # Fallback for the older malformed filename
        pattern = f"results/stage_ct_imagenet_to_dtd_resnet18_seed{seed}*.json"
        matches = glob.glob(pattern)

    if not matches:
        raise FileNotFoundError(f"No result found for seed {seed}")

    path = matches[0]

    print(f"Seed {seed}: {os.path.basename(path)}")

    with open(path, "r") as f:
        r = json.load(f)

    accs.append(r["accuracy"])
    val_accs.append(r["best_val_acc"])
    betas.append(r["stage_betas"])


accs = np.array(accs)
val_accs = np.array(val_accs)
betas = np.array(betas)


print("\n==============================")
print("DTD 4-PARAMETER SW-CT RESULTS")
print("==============================")

print("\nTest accuracies:")
for seed, acc in zip(seeds, accs):
    print(f"Seed {seed}: {acc:.4f}%")

print(f"\nMean test accuracy: {accs.mean():.2f}%")
print(f"Std test accuracy:  {accs.std():.2f}")


print("\nBest validation accuracies:")
for seed, acc in zip(seeds, val_accs):
    print(f"Seed {seed}: {acc:.4f}%")

print(f"\nMean best validation accuracy: {val_accs.mean():.2f}%")


print("\nStage betas:")
for seed, beta in zip(seeds, betas):
    print(
        f"Seed {seed}:",
        [round(x, 4) for x in beta]
    )

print("\nMean stage betas:")
print(np.round(betas.mean(axis=0), 4))

print("\nStd stage betas:")
print(np.round(betas.std(axis=0), 4))


print("\nPer-stage summary:")
for i in range(4):
    print(
        f"Stage {i+1}: "
        f"{betas[:, i].mean():.4f} ± "
        f"{betas[:, i].std():.4f}"
    )

Seed 42: stage_ct_imagenet_to_dtd_resnet18_seed42_initbeta0.78_betalr0.01.json
Seed 43: stage_ct_imagenet_to_dtd_resnet18_seed43_initbeta0.78_betalr0.01.json
Seed 44: stage_ct_imagenet_to_dtd_resnet18_seed44_initbeta0.78_betalr0.01.json

DTD 4-PARAMETER SW-CT RESULTS

Test accuracies:
Seed 42: 62.4468%
Seed 43: 62.8191%
Seed 44: 62.0213%

Mean test accuracy: 62.43%
Std test accuracy:  0.33

Best validation accuracies:
Seed 42: 61.4362%
Seed 43: 61.8085%
Seed 44: 62.0745%

Mean best validation accuracy: 61.77%

Stage betas:
Seed 42: [0.8744, 0.9224, 0.8811, 0.7103]
Seed 43: [0.8846, 0.9196, 0.88, 0.7028]
Seed 44: [0.8919, 0.914, 0.8785, 0.7047]

Mean stage betas:
[0.8836 0.9187 0.8799 0.7059]

Std stage betas:
[0.0072 0.0035 0.0011 0.0032]

Per-stage summary:
Stage 1: 0.8836 ± 0.0072
Stage 2: 0.9187 ± 0.0035
Stage 3: 0.8799 ± 0.0011
Stage 4: 0.7059 ± 0.0032


### DTD Results Summary

Across three seeds:

- Baseline: **62.80 ± 0.42%**
- S-CT: **62.52 ± 0.36%**
- SW-CT: **62.43 ± 0.33%**

Unlike Beans, curvature tuning does not improve average test accuracy over the baseline on DTD.

However, SW-CT learns a highly consistent stage-wise curvature pattern across seeds:

- Stage 1: β ≈ 0.884
- Stage 2: β ≈ 0.919
- Stage 3: β ≈ 0.880
- Stage 4: β ≈ 0.706

The same qualitative pattern observed on Beans is visible again: Stage 2 tends to retain the largest β, while Stage 4 learns a substantially lower β.

This suggests that the distribution of useful curvature across network depth may contain a repeatable structure even when the overall accuracy benefit varies by dataset. Additional datasets are required before treating this as a general conclusion.

# From Learned SW-CT to Formula-SWCT Calibration

The learned SW-CT experiments above evaluate whether four stage-specific curvature parameters can be optimized jointly with the downstream classifier.

The later phase of this research investigates a different objective: predicting the four stage-wise beta values directly before downstream training.

For this final Formula-SWCT setting, beta values are not optimized on the target dataset. Instead, they are predicted from measurable properties of the frozen pretrained network and remain fixed throughout downstream classifier training.

Because the deployment setting uses fixed beta values, the calibration targets for Formula-SWCT are derived from fixed-beta validation experiments rather than directly using the jointly learned SW-CT beta values.

## Stage-Isolated Fixed-Beta Calibration

To construct a calibration target compatible with the final Formula-SWCT deployment setting, each ResNet-18 stage is examined independently using fixed beta values.

For each stage, candidate beta values are evaluated while the other stages remain unchanged. Validation accuracy is used to identify the preferred fixed curvature value for that stage.

The calibration selection rule is:

> Choose the smallest beta achieving the maximum validation accuracy among the evaluated stage-isolated beta values.

For DTD, this procedure produces the final Formula-SWCT calibration target:

**[0.95, 0.95, 0.99, 0.99]**

corresponding to:

- Stage 1: **0.95**
- Stage 2: **0.95**
- Stage 3: **0.99**
- Stage 4: **0.99**

These values are calibration targets for the Formula-SWCT predictor and should not be interpreted as jointly learned SW-CT parameters.

## Role of DTD in Formula-SWCT

DTD is one of three development datasets used to fit the final Formula-SWCT predictor:

- Beans
- DTD
- Flowers102

For each dataset and ResNet-18 stage, five descriptors are measured from the frozen pretrained network:

1. log activation variance,
2. activation sparsity,
3. log mean absolute activation,
4. log class-separation statistic,
5. normalized stage depth.

These descriptors are paired with the stage-isolated fixed-beta calibration targets.

For DTD, the target vector is:

**[0.95, 0.95, 0.99, 0.99]**

Statistics are collected across seeds 42, 43, and 44. The combined calibration data from Beans, DTD, and Flowers102 are then used to fit the ridge-regression Formula-SWCT predictor.

Once fitted, the predictor is frozen before evaluation on additional target datasets.

## DTD Calibration Summary

DTD serves two roles in the development of Stage-Wise Curvature Tuning:

**Learned SW-CT evaluation:**  
The original experiments test whether four stage-specific curvature parameters improve transfer performance when optimized jointly with the downstream classifier.

**Formula-SWCT calibration:**  
The later fixed-beta experiments provide stage-wise targets compatible with the predict-first, train-once deployment protocol.

Final DTD calibration target:

**[0.95, 0.95, 0.99, 0.99]**

DTD is therefore treated as a Formula-SWCT development/calibration dataset rather than as an independent external generalization benchmark.